<a href="https://colab.research.google.com/github/Mahammad-Haroon/Data-Analysis-Projects/blob/main/E_Commerce_Outlier_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# E-Commerce Order Data - Outlier Analysis

This notebook identifies, analyses, and treats outliers in e-commerce order data using Z-score and IQR methods.

## 1. Dataset Creation

In [1]:
import pandas as pd
import numpy as np
from scipy import stats

np.random.seed(99)
n = 800

df = pd.DataFrame({
    "purchase_value": np.append(
        np.random.exponential(scale=120, size=792),
        [7200, 9500, 11000, 8800, 10200, 12500, 6900, 9100]
    ),

    "delivery_days": np.append(
        np.abs(np.random.normal(loc=6, scale=2, size=793)),
        [44, 58, 63, 50, 47, 55, 7]
    ),

    "review_score": np.clip(
        np.append(
            np.random.normal(loc=4.0, scale=0.6, size=799),
            [1]
        ),
        1,
        5
    ).round(1)
})

print(df.shape)
print(df.describe())

(800, 3)
       purchase_value  delivery_days  review_score
count      800.000000     800.000000    800.000000
mean       222.260039       6.474309      3.978375
std        948.594784       4.550191      0.593262
min          0.002088       0.196098      1.000000
25%         36.898998       4.740409      3.600000
50%         90.173587       6.214954      4.000000
75%        181.334721       7.570977      4.400000
max      12500.000000      63.000000      5.000000


## Q1 - Profile and Choose Detection Method

In [2]:
for col in ["purchase_value", "delivery_days", "review_score"]:
    desc = df[col].describe()

    mean = desc["mean"]
    median = desc["50%"]

    percent_diff = abs(mean - median) / median * 100

    print(f"\nColumn: {col}")
    print(f"Mean: {mean:.2f}")
    print(f"Median: {median:.2f}")
    print(f"Percentage difference: {percent_diff:.2f}%")

    if percent_diff < 10:
        print("Decision: Use Z-score because the distribution is relatively symmetric.")
    else:
        print("Decision: Use IQR because the distribution appears skewed.")


Column: purchase_value
Mean: 222.26
Median: 90.17
Percentage difference: 146.48%
Decision: Use IQR because the distribution appears skewed.

Column: delivery_days
Mean: 6.47
Median: 6.21
Percentage difference: 4.17%
Decision: Use Z-score because the distribution is relatively symmetric.

Column: review_score
Mean: 3.98
Median: 4.00
Percentage difference: 0.54%
Decision: Use Z-score because the distribution is relatively symmetric.


## Q2 - Detect Delivery Day Outliers Using Z-score

In [3]:
z = stats.zscore(df["delivery_days"])

delivery_outlier_mask = abs(z) > 3

print("Number of delivery_days outliers:", delivery_outlier_mask.sum())

print("\nFlagged delivery_days values:")
print(df.loc[delivery_outlier_mask, "delivery_days"])

Number of delivery_days outliers: 6

Flagged delivery_days values:
793    44.0
794    58.0
795    63.0
796    50.0
797    47.0
798    55.0
Name: delivery_days, dtype: float64


### Observation

The Z-score method identified unusually large delivery times as outliers.
These values are far from the normal delivery-time pattern and are potential anomalies.

## Q3 - Detect Purchase Value Outliers Using IQR

In [4]:
Q1 = df["purchase_value"].quantile(0.25)
Q3 = df["purchase_value"].quantile(0.75)

IQR = Q3 - Q1

lower_fence = Q1 - 1.5 * IQR
upper_fence = Q3 + 1.5 * IQR

purchase_outlier_mask = (
    (df["purchase_value"] < lower_fence) |
    (df["purchase_value"] > upper_fence)
)

print(f"Q1: {Q1:.2f}")
print(f"Q3: {Q3:.2f}")
print(f"IQR: {IQR:.2f}")
print(f"Lower Fence: {lower_fence:.2f}")
print(f"Upper Fence: {upper_fence:.2f}")

print("\nNumber of purchase_value outliers:",
      purchase_outlier_mask.sum())

print("\nFlagged purchase_value rows:")
print(df.loc[purchase_outlier_mask, ["purchase_value"]])

Q1: 36.90
Q3: 181.33
IQR: 144.44
Lower Fence: -179.75
Upper Fence: 397.99

Number of purchase_value outliers: 46

Flagged purchase_value rows:
     purchase_value
8        560.395762
16       437.757511
47       414.072407
62       449.842273
104      644.891199
137      406.016583
213      525.037418
223      546.868518
243      401.822466
252      545.441443
253      453.170321
281      884.239258
301      449.582406
310      399.513071
345      514.762691
361      859.463624
379      419.989614
400      613.130126
423      417.616309
434      488.208236
441      761.177298
458      459.711114
495      715.083886
497      557.894765
523      464.459238
547      512.509202
557      638.935515
583      438.250843
619      478.784967
645      602.915459
650      402.106188
697      876.121631
698      549.811003
706      539.591194
708      595.608205
709      661.161665
728      415.250873
732      719.968811
792     7200.000000
793     9500.000000
794    11000.000000
795     8800.0000

## Q4 - Remove Delivery Day Outliers

In [5]:
print("Shape before removal:", df.shape)

df_clean = df.loc[~delivery_outlier_mask].copy()

print("Shape after removal:", df_clean.shape)

print("\nBefore removal:")
print(df["delivery_days"].describe())

print("\nAfter removal:")
print(df_clean["delivery_days"].describe())

Shape before removal: (800, 3)
Shape after removal: (794, 3)

Before removal:
count    800.000000
mean       6.474309
std        4.550191
min        0.196098
25%        4.740409
50%        6.214954
75%        7.570977
max       63.000000
Name: delivery_days, dtype: float64

After removal:
count    794.000000
mean       6.123989
std        2.038548
min        0.196098
25%        4.737017
50%        6.194851
75%        7.543245
max       13.534982
Name: delivery_days, dtype: float64


## Q5 - Cap Purchase Value Outliers

In [6]:
df["purchase_value_capped"] = df["purchase_value"].clip(
    lower=lower_fence,
    upper=upper_fence
)

print("Original purchase_value:")
print(df["purchase_value"].describe())

print("\nCapped purchase_value:")
print(df["purchase_value_capped"].describe())

print("\nOriginal maximum:",
      df["purchase_value"].max())

print("Capped maximum:",
      df["purchase_value_capped"].max())

print("Upper fence:",
      upper_fence)

Original purchase_value:
count      800.000000
mean       222.260039
std        948.594784
min          0.002088
25%         36.898998
50%         90.173587
75%        181.334721
max      12500.000000
Name: purchase_value, dtype: float64

Capped purchase_value:
count    800.000000
mean     125.117912
std      111.347550
min        0.002088
25%       36.898998
50%       90.173587
75%      181.334721
max      397.988304
Name: purchase_value_capped, dtype: float64

Original maximum: 12500.0
Capped maximum: 397.9883039754379
Upper fence: 397.9883039754379


## Q6 - High Value Order Flag and Review Analysis

In [7]:
df["is_high_value_order"] = (
    df["purchase_value"] > upper_fence
)

print(df["is_high_value_order"].value_counts())

is_high_value_order
False    754
True      46
Name: count, dtype: int64


In [8]:
review_comparison = (
    df.groupby("is_high_value_order")["review_score"]
      .mean()
)

print(review_comparison)

is_high_value_order
False    3.979045
True     3.967391
Name: review_score, dtype: float64
